# 1. Header, Citations, Purpose & Topic Bridges

### Theoretical Significance & Research Lineage
Discrete grids (pixels, voxels) scale poorly and cannot represent continuous physical fields with sub-pixel fidelity. Sitzmann et al. (2020) introduced **SIREN**, showing that periodic sinusoidal activations preserve spatial gradients and solve Poisson boundary value problems. Mildenhall et al. (2020) pioneered **NeRF** (Neural Radiance Fields), using volume rendering quadrature to synthesize 3D scenes from 2D photos. Müller et al. (2022) accelerated neural fields with **Instant-NGP** multi-resolution hash encodings. Kerbl et al. (2023) developed **3D Gaussian Splatting (3DGS)**, replacing implicit MLPs with differentiable 3D Gaussians for real-time 100+ FPS rendering.

### Purpose & Target Architecture
This notebook constructs from first principles:
1. SIREN sinusoidal coordinate network layer with Sitzmann initialization.
2. NeRF numerical volume rendering quadrature (transmittance, optical depth, color composite).
3. 3DGS 2D projected Gaussian alpha evaluation.

# 2. Mathematical Foundations & Governing Equations

### SIREN Periodic Activation
Parameterizing coordinate mappings with frequency scaling $\omega_0$:
$$\Phi(x) = \sin(\omega_0 (W x + b))$$

### NeRF Volume Rendering Quadrature
For ray samples $i = 1 \dots S$ with optical depth $\tau_i = \sigma_i \delta_i$:
$$\hat{C} = \sum_{i=1}^S T_i (1 - e^{-\tau_i}) c_i, \quad T_i = \exp\left(-\sum_{j=1}^{i-1} \tau_j\right)$$

### 3DGS Projected Alpha Splatting
Given 2D screen coordinate $x$, center $\mu$, and inverse covariance $\Sigma^{-1}$:
$$\alpha(x) = \exp\left(-\frac{1}{2} (x - \mu)^T \Sigma^{-1} (x - \mu)\right)$$

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import math
import torch
import torch.nn.functional as F

torch.manual_seed(42)

# Synthetic ray setup: 2 rays, 8 depth samples per ray, 3 RGB color channels
B, S = 2, 8
densities = torch.abs(torch.randn(B, S)) * 2.0
colors = torch.sigmoid(torch.randn(B, S, 3))
deltas = torch.full((B, S), 0.125)

# 2D coordinates for SIREN / 3DGS
coords_2d = torch.tensor([[0.0, 0.0], [0.2, -0.3], [0.5, 0.5], [-0.4, 0.8]])
print(f"Volume rendering setup: rays={B}, samples per ray={S}, deltas={deltas[0, 0].item():.3f}")

In [ ]:
# 4. First-Principles Implementation & Native Parity

def siren_layer(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor, omega_0: float = 30.0) -> torch.Tensor:
    return torch.sin(omega_0 * (x @ weight + bias))

def volume_rendering_quadrature(densities: torch.Tensor, colors: torch.Tensor, deltas: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    b, s = densities.shape
    tau = F.relu(densities) * deltas
    alpha = 1.0 - torch.exp(-tau)

    cum_tau = torch.cumsum(tau, dim=-1)
    zeros = torch.zeros(b, 1, device=densities.device)
    prev_tau = torch.cat([zeros, cum_tau[:, :-1]], dim=-1)
    transmittance = torch.exp(-prev_tau)

    weights = (transmittance * alpha).unsqueeze(-1)
    rendered_color = torch.sum(weights * colors, dim=1)
    accum_opacity = torch.sum(weights, dim=1)
    return rendered_color, accum_opacity

def gaussian_splat_2d_eval(x: torch.Tensor, mu: torch.Tensor, cov_inv: torch.Tensor) -> torch.Tensor:
    delta = x - mu
    proj = delta @ cov_inv
    mahalanobis = torch.sum(proj * delta, dim=-1, keepdim=True)
    return torch.exp(-0.5 * mahalanobis)

# Execute forward passes
rendered_color, accum_opacity = volume_rendering_quadrature(densities, colors, deltas)
mu = torch.tensor([[0.0, 0.0]])
cov_inv = torch.tensor([[4.0, 0.0], [0.0, 4.0]])
splat_alpha = gaussian_splat_2d_eval(coords_2d, mu, cov_inv)

print(f"Rendered RGB color: ray 0 = {rendered_color[0].tolist()}")
print(f"Accumulated ray opacity: {accum_opacity.squeeze(-1).tolist()}")
print(f"3DGS Gaussian alphas: {splat_alpha.squeeze(-1).tolist()}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification

# Invariant 1: Physical conservation of light: total transmittance + opacity <= 1.0
assert torch.all(accum_opacity <= 1.0 + 1e-6), "Accumulated opacity cannot physically exceed 1.0"
assert torch.all(accum_opacity >= 0.0), "Accumulated opacity must be non-negative"

# Invariant 2: Zero density produces zero opacity and zero color
zero_density = torch.zeros_like(densities)
empty_color, empty_opacity = volume_rendering_quadrature(zero_density, colors, deltas)
assert torch.allclose(empty_color, torch.zeros_like(empty_color), atol=1e-6), "Zero density must yield zero rendered color"
assert torch.allclose(empty_opacity, torch.zeros_like(empty_opacity), atol=1e-6), "Zero density must yield zero opacity"

# Invariant 3: 3DGS Gaussian peak at center x == mu is strictly 1.0
center_alpha = gaussian_splat_2d_eval(mu, mu, cov_inv)
assert torch.allclose(center_alpha, torch.tensor([[1.0]]), atol=1e-6), "Gaussian splat alpha at center must equal 1.0"
print("All INR, NeRF, and 3DGS mathematical invariants verified successfully!")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

# Benchmark NeRF volume integration vs 3DGS evaluation
start = time.perf_counter()
for _ in range(500):
    _ = volume_rendering_quadrature(densities, colors, deltas)
nerf_quad_time = (time.perf_counter() - start) / 500.0

start = time.perf_counter()
for _ in range(500):
    _ = gaussian_splat_2d_eval(coords_2d, mu, cov_inv)
splat_time = (time.perf_counter() - start) / 500.0

print(f"NeRF volume rendering quadrature latency: {nerf_quad_time*1e6:.1f} µs")
print(f"3DGS Gaussian alpha evaluation latency: {splat_time*1e6:.1f} µs")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation

# Pathological Misuse: Unconstrained negative densities in NeRF
def unconstrained_volume_rendering_bad(densities: torch.Tensor, colors: torch.Tensor, deltas: torch.Tensor):
    # Without ReLU on density, negative values cause exponential light amplification
    tau = densities * deltas
    alpha = 1.0 - torch.exp(-tau)
    cum_tau = torch.cumsum(tau, dim=-1)
    zeros = torch.zeros(densities.shape[0], 1)
    prev_tau = torch.cat([zeros, cum_tau[:, :-1]], dim=-1)
    transmittance = torch.exp(-prev_tau)
    weights = (transmittance * alpha).unsqueeze(-1)
    return torch.sum(weights * colors, dim=1), torch.sum(weights, dim=1)

neg_densities = torch.full((1, 4), -5.0)
dummy_colors = torch.ones(1, 4, 3)
dummy_deltas = torch.full((1, 4), 0.5)

bad_color, bad_opacity = unconstrained_volume_rendering_bad(neg_densities, dummy_colors, dummy_deltas)
good_color, good_opacity = volume_rendering_quadrature(neg_densities, dummy_colors, dummy_deltas)

print(f"Pathological unconstrained opacity: {bad_opacity.item():.4f} (Violates physics, negative opacity)")
print(f"Corrected ReLU opacity: {good_opacity.item():.4f} (Strictly physical zero opacity)")

# 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

### Core Insights
1. **Continuity Eliminates Discretization Limits**: INRs decouple spatial query resolution from model memory footprint, admitting continuous analytical derivatives everywhere.
2. **Physical Volume Rendering**: The Beer-Lambert law integration equation translates continuous 3D radiance fields into camera pixels differentiably.
3. **Explicit vs Implicit Tradeoff**: While NeRF requires evaluating neural networks hundreds of times per camera ray, 3DGS parameterizes the scene with explicit anisotropic Gaussians, delivering orders-of-magnitude faster rasterization.